# Generare musica folk irlandese con una LSTM

In questo lab insegniamo a una rete neurale a scrivere musica. L'idea è più semplice di quanto sembri: il nostro dataset è una raccolta di canzoni folk irlandesi scritte in **notazione ABC**, che è puro testo. Possiamo quindi trattare la generazione musicale come un problema di **language modeling** a livello di carattere: data una sequenza di caratteri, predire il successivo. Una volta che il modello è bravo in questo, possiamo lasciargli "allucinare" canzoni completamente nuove, un carattere alla volta, e poi sintetizzarle in audio.

Lungo il percorso costruiremo una **LSTM character-level** in PyTorch, aggiungeremo un vero **split train/validation**, **early stopping** e **checkpointing**, e tracceremo ogni run con **Comet ML**.

## Eseguire questo notebook su Kaggle

Questo notebook è pensato per girare su **Kaggle**, che ci offre una GPU gratuita e un modo pulito per conservare le API key. Per impostarlo:

1. Crea un nuovo Notebook su [Kaggle](https://www.kaggle.com/code) e importa questo file (File > Import Notebook);
2. Nelle impostazioni del notebook, imposta **Accelerator** su una GPU (ad esempio una T4 o una P100) e assicurati che **Internet** sia abilitato;
3. Crea un account gratuito su [Comet](https://www.comet.com), che useremo per tracciare gli esperimenti, e copia la tua **API key** dalle impostazioni dell'account;
4. Tornato su Kaggle, apri **Add-ons > Secrets**, crea una secret con label `COMET_API_KEY`, incolla la tua key come valore e collegala a questo notebook.

La prima cella legge la key tramite `kaggle_secrets`, quindi la key non compare mai nel codice. Se esegui il notebook altrove (Colab, macchina locale), basta sostituire quel blocco con il tuo modo di caricare la key.

Una nota sulle dipendenze: il pacchetto `mitdeeplearning` è una piccola libreria di utility del corso open di deep learning del MIT. Qui la usiamo solo per scaricare il dataset delle canzoni e per un paio di helper di plotting e audio.

In [ ]:
# importiamo i pacchetti

!pip install comet_ml > /dev/null 2>&1
import comet_ml
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
COMET_API_KEY = user_secrets.get_secret("COMET_API_KEY")

import torch
import torch.nn as nn
import torch.optim as optim

!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
from scipy.io.wavfile import write
!apt-get install abcmidi timidity > /dev/null 2>&1

from IPython import display
import matplotlib.pyplot as plt

assert COMET_API_KEY != "", "Please insert your Comet API Key"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Il dataset

Carichiamo migliaia di canzoni folk irlandesi e le uniamo in un'unica gigantesca stringa. Da quella stringa estraiamo il **vocabolario**: l'insieme ordinato di ogni carattere unico che compare (lettere, cifre, punteggiatura, a capo). Questo è l'intero "alfabeto" che il nostro modello conoscerà. Il modello non sa cosa siano una nota o un accordo: vede solo caratteri, e dovrà scoprire da solo la struttura della musica.

In [ ]:
songs = mdl.lab1.load_training_data()
songs_joined = "\n\n".join(songs)
vocab = sorted(set(songs_joined))

## Dai caratteri ai numeri

Le reti neurali non mangiano testo, mangiano numeri. Costruiamo quindi due tabelle di lookup: `char2idx` mappa ogni carattere in un id intero univoco, e `idx2char` fa l'inverso. Insieme ci permettono di muoverci avanti e indietro tra il mondo del testo e il mondo dei tensori.

In [ ]:
# funzioni per passare da carattere a id numerico e viceversa, le useremo come tabella di lookup
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

Con la tabella di lookup pronta, vettorizzare una stringa è solo questione di sostituire ogni carattere con il suo id. Lo applichiamo all'intero dataset, ottenendo un lungo array NumPy di interi.

In [ ]:
def vectorize_string(string):
    vector = np.array([char2idx[char] for char in string])
    return vector

vectorized_songs = vectorize_string(songs_joined)
assert isinstance(vectorized_songs, np.ndarray)

## Costruire i batch di training

Ecco il trucco che fa funzionare tutto il lab. Per ogni esempio di training prendiamo una fetta casuale del dataset di lunghezza `seq_length` come **input**, e la stessa fetta **spostata di una posizione a destra** come **target**. Perché? Perché il compito che vogliamo imparare è "predici il prossimo carattere": in ogni posizione, la risposta corretta è esattamente il carattere che segue.

Ad esempio, se il testo è `Hello`, l'input è `Hell` e il target è `ello`: dato `H` predici `e`, dato `He` predici `l`, e così via. Impiliamo `batch_size` di queste fette casuali insieme, così la GPU può processarle in parallelo.

In [ ]:
def get_batch(vectorized_songs, seq_length, batch_size):
    # calcoliamo l'indice massimo della stringa vettorizzata per evitare accessi fuori dai limiti
    n = vectorized_songs.shape[0] - 1
    # scegliamo indici casuali come punti di partenza dei batch
    idx = np.random.choice(n - seq_length, batch_size)
    # prendiamo un numero di fette di input pari a batch_size
    input_batch = [vectorized_songs[i: i + seq_length] for i in idx]
    # stessa logica per il target, ma spostato di una posizione a destra, perche' e' la predizione
    output_batch = [vectorized_songs[i+ 1: i + seq_length + 1] for i in idx]

    x_batch = torch.tensor(input_batch, dtype=torch.long).to(device)
    y_batch = torch.tensor(output_batch, dtype=torch.long).to(device)

    return x_batch, y_batch

test_args = (vectorized_songs, 2, 10)
x_batch, y_batch = get_batch(*test_args)
print(vectorized_songs.shape)
print(x_batch.shape)
print(y_batch.shape)
example_idx = 0
print("X (Input) :", x_batch[example_idx].tolist())
print("Y (Target):", y_batch[example_idx].tolist())

## Il modello: Embedding, LSTM, Linear

La nostra rete è una pila di tre pezzi:

- **Embedding**: trasforma ogni id di carattere in un vettore denso di `embedding_dim` numeri. Invece di un intero senza significato, ogni carattere riceve una rappresentazione appresa, e caratteri simili possono finire con vettori simili;
- **LSTM**: il cuore del modello. Legge la sequenza un passo alla volta portandosi dietro uno **stato** interno, una sorta di memoria che le permette di ricordare informazioni di molti caratteri fa (essenziale per la musica: il finale di una frase dipende da come è iniziata). Impiliamo `num_layers` layer LSTM con dropout in mezzo per ridurre l'overfitting;
- **Linear**: la testa finale. Proietta l'output della LSTM in un vettore con un punteggio (**logit**) per ogni carattere del vocabolario: la nostra predizione su cosa viene dopo.

Il metodo `init_hidden` costruisce uno stato iniziale azzerato, e `forward` può opzionalmente restituire lo stato finale, che ci servirà più avanti durante la generazione per mantenere viva la "memoria" tra un carattere e il successivo.

In [ ]:
# definiamo il nostro modello di rete neurale ricorrente (RNN)

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super(LSTMModel, self).__init__()

        self.num_layers = num_layers
        self.hidden_size = hidden_size
        # per convertire il testo in un vettore di numeri
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # la LSTM analizza i vettori di input e mantiene uno stato interno, permettendo alla rete di ricordare informazioni a lungo termine
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size, num_layers=num_layers, dropout=0.2, batch_first=True)
        # riceve in input l'hidden state e lo trasforma nel vettore con il punteggio per ogni carattere
        self.fc = nn.Linear(hidden_size, vocab_size)

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
            torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))

    def forward(self, x, state=None, return_state=False):
        x = self.embedding(x)

        if state is None:
          state = self.init_hidden(x.size(0), x.device)
        out, state = self.lstm(x, state)

        out = self.fc(out)
        return out if not return_state else (out, state)

## La loss function

Questo è un problema di classificazione: in ogni posizione, "quale dei caratteri del vocabolario viene dopo?". Usiamo quindi la **cross entropy**. L'unica sottigliezza è la forma: il modello produce un tensore 3D (batch, sequenza, vocabolario), mentre `CrossEntropyLoss` ne vuole uno 2D. Semplicemente appiattiamo batch e sequenza insieme, così ogni posizione di ogni sequenza conta come una predizione indipendente.

In [ ]:
cross_entropy = nn.CrossEntropyLoss()
def compute_loss(labels, logits):
    batched_labels = labels.view(-1)
    batched_logits = logits.view(-1, logits.size(-1))
    loss = cross_entropy(batched_logits, batched_labels)
    return loss

## Split train/validation

Teniamo da parte l'ultimo 10% dei dati come **validation set**. Il modello non ci si allena mai, quindi misurare la loss lì ci dice quanto bene generalizza su musica che non ha mai visto. Se la training loss continua a scendere mentre la validation loss inizia a salire, il modello sta andando in **overfitting**: sta memorizzando le canzoni di training invece di imparare lo stile.

In [ ]:
split = int(0.9 * len(vectorized_songs))
train_data = vectorized_songs[:split]
val_data = vectorized_songs[split:]

La funzione di validation loss media la loss su alcuni batch casuali di validation. Nota i due interruttori di sicurezza: `model.eval()` disabilita il dropout, e `torch.no_grad()` disabilita il tracciamento dei gradienti, perché qui vogliamo solo misurare, non imparare. Alla fine torniamo in `model.train()`.

In [ ]:
def compute_val_loss(num_batches=20):
    model.eval()
    with torch.no_grad():
        losses = []
        for _ in range(num_batches):
            x, y = get_batch(val_data, params["seq_length"], params["batch_size"])
            y_hat = model(x)
            losses.append(compute_loss(y, y_hat).item())
    model.train()
    return np.mean(losses)

Un piccolo helper per plottare in tempo reale le curve di training e validation durante l'addestramento. La validation loss viene calcolata ogni 10 iterazioni, quindi scaliamo il suo asse x di conseguenza.

In [ ]:
def plot_losses(history, val_history):
    display.clear_output(wait=True)
    plt.figure(figsize=(10, 4))
    plt.plot(history, label="train loss")
    val_iters = [i * 10 for i in range(len(val_history))]
    plt.plot(val_iters, val_history, label="val loss")
    plt.xlabel("Iterations")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

## Iperparametri

Tutte le manopole in un unico posto:

- `seq_length = 300`: quanti caratteri di contesto il modello vede per esempio;
- `batch_size = 256`: quante sequenze processiamo in parallelo;
- `embedding_dim = 256` e `hidden_size = 1024`: la capacità del modello;
- `num_layers = 2`: due layer LSTM impilati;
- `learning_rate = 5e-3`: la dimensione del passo dell'ottimizzatore.

Sentiti libero di sperimentare: sono esattamente i parametri con cui vale la pena giocare per vedere come reagiscono le curve di loss.

In [ ]:
vocab_size = len(vocab)

params = dict(
    num_training_iteractions = 500,
    batch_size = 256,
    seq_length = 300,
    learning_rate = 5e-3,
    embedding_dim = 256,
    hidden_size = 1024,
    num_layers = 2
)

torch.cuda.empty_cache()

## Tracciamento degli esperimenti con Comet

Ogni run di training viene loggata su **Comet ML**: iperparametri, curve di loss, e alla fine perfino i file audio generati. È un'abitudine che vale la pena costruire presto: quando inizi a modificare gli iperparametri, poter confrontare le run fianco a fianco vale oro.

In [ ]:
def create_experiment():
  if 'experiment' in locals():
    experiment.end()

  experiment = comet_ml.Experiment(api_key=COMET_API_KEY, project_name="irish-folk-music-generation")

  for param, value in params.items():
    experiment.log_parameter(param, value)

  return experiment

## Il training step

Istanziamo il modello, lo spostiamo sulla GPU e usiamo **Adam** come ottimizzatore. Un singolo training step è il classico loop PyTorch: azzeriamo i gradienti con `zero_grad()`, eseguiamo il **forward pass**, calcoliamo la loss, chiamiamo `loss.backward()` per la backpropagation e lasciamo che `optimizer.step()` aggiorni i pesi.

In [ ]:
model = LSTMModel(vocab_size, params["embedding_dim"], params["hidden_size"], num_layers=params["num_layers"])
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])

def train_step(x, y):
    model.train()
    optimizer.zero_grad()
    x = x.to(device)
    y = y.to(device)
    y_hat = model(x)
    loss = compute_loss(y, y_hat)
    loss.backward()
    optimizer.step()
    return loss

Prepariamo una directory per i **checkpoint**: ogni volta che la validation loss migliora salveremo lì i pesi del modello, così alla fine potremo ripristinare la versione migliore, non solo l'ultima.

In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt")
os.makedirs(checkpoint_dir, exist_ok=True)

## Il training loop, con early stopping

Il loop mette insieme tutto: prendi un batch, fai un training step, logga la loss. Ogni 10 iterazioni misuriamo anche la **validation loss**, ed è qui che vive la logica interessante:

- se la validation loss è migliorata, salviamo un checkpoint e azzeriamo il contatore di pazienza;
- se non migliora per `patience = 5` controlli consecutivi, fermiamo l'addestramento in anticipo (**early stopping**). Andare avanti significherebbe solo overfitting.

Quando il loop termina, ricarichiamo il checkpoint migliore. In questo modo il modello che teniamo è quello che ha generalizzato meglio, non quello dell'ultima iterazione.

In [ ]:
history = []
val_history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel="Iterations", ylabel="Loss")
experiment = create_experiment()

best_val_loss = float('inf')
patience = 5
no_improve = 0

if hasattr(tqdm, '_instances'): tqdm._instances.clear()
    
for iter in tqdm(range(params["num_training_iteractions"]), unit="it", leave=True):
    x_batch, y_batch = get_batch(train_data, params["seq_length"], params["batch_size"])

    loss = train_step(x_batch, y_batch)

    experiment.log_metric("loss", loss.item(), step=iter)

    history.append(loss.item())
    plot_losses(history, val_history)

    if iter % 10 == 0:
        val_loss = compute_val_loss()
        val_history.append(val_loss)
        experiment.log_metric("val_loss", val_loss, step=iter)
    
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve = 0
            torch.save(model.state_dict(), checkpoint_prefix)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at iter {iter}, best val_loss: {best_val_loss:.4f}")
                break

experiment.flush()
model.load_state_dict(torch.load(checkpoint_prefix))

## Generare nuova musica

Ora la parte divertente. La generazione è **autoregressiva**: diamo al modello una stringa di partenza, lui predice una distribuzione di probabilità sul prossimo carattere, ne scegliamo uno, lo appendiamo all'input e ripetiamo.

Un dettaglio chiave: non scegliamo sempre il carattere più probabile (sarebbe greedy e ripetitivo). Invece **campioniamo** dalla distribuzione con `torch.multinomial`, così il modello può sorprenderci. Nota anche che ci portiamo dietro lo `state` della LSTM a ogni passo: è la memoria del modello di tutto ciò che è stato generato finora.

In [ ]:
def generate_text(model, start_string, generation_length=1000):

  input_idx = [char2idx[s] for s in start_string]
  input_idx = torch.tensor([input_idx], dtype=torch.long).to(device)

  state = model.init_hidden(input_idx.size(0), device)

  text_generated = []
  tqdm._instances.clear()
  with torch.no_grad():
    for i in tqdm(range(generation_length)):
      predictions, state = model(input_idx, state, return_state=True)
      predictions = predictions.squeeze(0)

      input_idx = torch.multinomial(torch.softmax(predictions, dim=-1), num_samples=1)

      text_generated.append(idx2char[input_idx].item())

  return (start_string + ''.join(text_generated))

## Dal testo all'audio

Il testo generato dovrebbe contenere canzoni valide in notazione ABC. Estraiamo ogni frammento di canzone ben formato, lo sintetizziamo in una waveform, lo riproduciamo inline, lo salviamo come file `.wav` e lo logghiamo su Comet, così possiamo ascoltare le composizioni del nostro modello direttamente dalla pagina dell'esperimento.

Non tutti i frammenti saranno sintatticamente validi, ed è normale: il modello ha imparato il formato solo dagli esempi. Ascolta qualche output, poi prova a giocare con gli iperparametri o con la lunghezza di generazione e osserva come cambia la musica.

In [ ]:
generated_text = generate_text(model, start_string="X", generation_length=1000)
generated_songs = mdl.lab1.extract_song_snippet(generated_text)

for i, song in enumerate(generated_songs):
  # sintetizziamo la waveform a partire dalla canzone
  waveform = mdl.lab1.play_song(song)

  # se e' una canzone valida (sintassi corretta), riproduciamola!
  if waveform:
    ipythondisplay.display(waveform)

    numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
    wav_file_path = f"output_{i}.wav"
    write(wav_file_path, 88200, numeric_data)

    # salviamo la canzone sull'interfaccia Comet, potrai accedervi da li'
    experiment.log_asset(wav_file_path)